# 2 — Fetch ATM Strike OHLC Data (Continuous Loop)

**Run frequency:** Every trading day, during market hours (09:15 – 15:30 IST)

**What this does:**
1. Logs in to Zerodha KiteConnect via automated TOTP
2. Starts a SparkSession
3. Reads instrument data from Parquet (written by Notebook 1)
4. Determines the ATM strike for BankNifty (live LTP or custom override)
5. Selects N strikes above/below ATM for the current expiry
6. Fetches 3-minute OHLC + OI history for each strike via KiteConnect API
7. Writes data to partitioned Parquet files
8. **Loops every minute** to keep pulling and appending the latest candle

**Runs in parallel with Notebook 3** (charts will auto-refresh as new data arrives)

---
### ⚙️ Configuration
Edit the `CONFIG` cell below to adjust parameters.

| Parameter | Default | Description |
|-----------|---------|-------------|
| `BANKNIFTY` | `True` | Pull BankNifty data |
| `NIFTY` | `False` | Pull Nifty data |
| `CUSTOM_STRIKE` | `0` | Override ATM strike (0 = live LTP) |
| `NUM_STRIKES` | `9` | N strikes each side of ATM |
| `PULL_NEXT_EXPIRY` | `False` | Also pull next weekly expiry |
| `NUM_DAYS_HISTORY` | `1` | Days of history per pull |
| `LOOP_INTERVAL_MIN` | `1` | Minutes between pulls |

In [22]:
# # ── Setup ──────────────────────────────────────────────────────────────────
import sys, os
# sys.path.insert(0, os.path.abspath('..'))
# os.chdir('..')  # Run from repo root so DataFiles/ paths resolve correctly

In [23]:
# ── CONFIG — edit these values ─────────────────────────────────────────────
BANKNIFTY         = True     # Pull BankNifty options data
NIFTY             = False    # Pull Nifty options data
CUSTOM_STRIKE     = 56500    # 0 = live LTP; set e.g. 56500 to override ATM
NUM_STRIKES       = 9        # How many ITM/OTM strikes each side of ATM
PULL_NEXT_EXPIRY  = False    # Also pull next weekly expiry
NUM_DAYS_HISTORY  = 4        # Calendar days of history per pull
LOOP_INTERVAL_MIN = 1        # Loop frequency in minutes
INTERVAL          = '3minute'

In [24]:
# ── Step 1: Login ──────────────────────────────────────────────────────────
from utils.kite_helpers import kite_login, get_spark_session

kite, kws, access_token = kite_login()
print('✅ Login successful')


2026-03-11 15:23:59,187 [INFO] utils.kite_helpers — Logging in as user: ZB4988
2026-03-11 15:23:59,397 [INFO] utils.kite_helpers — Generated TOTP pin for 2FA
2026-03-11 15:23:59,612 [INFO] utils.kite_helpers — Login successful — request_token: 82K24AKcNtfFm2ok6Zq350wprFgbDl5E
2026-03-11 15:23:59,813 [INFO] utils.kite_helpers — KiteConnect session established for user: ZB4988


✅ Login successful


In [25]:
# ── Step 2: Start SparkSession ─────────────────────────────────────────────
spark = get_spark_session(app_name='2_Fetch_Strikes_Data')
print('✅ Spark session ready')

✅ Spark session ready


In [26]:
# ── Step 3: Run continuous data pull loop ──────────────────────────────────
# This cell blocks until you interrupt the kernel (Kernel → Interrupt)
from utils.data_fetcher import run_data_loop

run_data_loop(
    kite                  = kite,
    spark                 = spark,
    banknifty             = BANKNIFTY,
    nifty                 = NIFTY,
    custom_strike         = CUSTOM_STRIKE,
    num_strikes           = NUM_STRIKES,
    pull_next_expiry      = PULL_NEXT_EXPIRY,
    num_days_history      = NUM_DAYS_HISTORY,
    interval              = INTERVAL,
    loop_interval_minutes = LOOP_INTERVAL_MIN,
)

Running data pull at 2026-03-11 15:31:43


2026-03-11 15:31:43,985 [INFO] utils.data_fetcher — Data pull | BNF ATM=56500 | NIFTY ATM=0 | from=2026-03-07 15:31:43.985198 | to=2026-03-11 15:31:43.985198


BankNifty ATM Strike = 56500 | Nifty ATM Strike = 0 | Data pulled at: 2026-03-11 15:31:43.985198
Pulling Current Expiry (2026-03-30) data…


2026-03-11 15:31:49,985 [ERROR] root — KeyboardInterrupt while sending command. 
Traceback (most recent call last):
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/socket.py", line 708, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt
2026-03-11 15:31:49,995 [INFO] py4j.clientserver — Closing down clientserver connection
2026-03-11 15:31:49,997 [INFO] utils.data_fetcher — Data loop stopped by user.
2026-03-11 15:31:49,997 [INFO] py4j.clientserv